# LLM 13: Hallucination & Guardrails

Hands-on:
1. Trigger a hallucination on purpose
2. Add grounding and watch it disappear
3. Add an abstention prompt
4. Self-critique pattern
5. Rule-based validator + policy classifier sketch
6. Exercises: measure wrong-confident rate; calibration check

Deps: `pip install anthropic openai`

## 1. Shared Caller

In [ ]:
import os

def call_llm(system, user, temperature=0):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(
            model='claude-sonnet-4-6', max_tokens=400, temperature=temperature,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model='gpt-4o-mini', temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '[no provider]'

## 2. Trigger a Hallucination

Ask a question about a fictional entity. A poorly-guarded model will confidently confabulate.

In [ ]:
q = 'What were the Q3 2025 revenue and CEO comments for Zyphor Technologies Inc?'
print(call_llm('You are a helpful financial analyst.', q))

## 3. Add Grounding

In [ ]:
context = '''<context>
No filings or press releases exist in the knowledge base for a company
named Zyphor Technologies Inc. The knowledge base was last refreshed
on 2026-04-15.
</context>'''

grounded = f'''{context}

<question>{q}</question>

<instructions>
Answer strictly from the context. If the context does not contain the answer,
respond with "I cannot answer from the provided context."
</instructions>'''

print(call_llm('You are a financial analyst. Follow instructions strictly.', grounded))

## 4. Abstention Prompt

In [ ]:
abstain_system = ('You are a careful analyst. If you do not know an answer with high confidence, '
                  'respond ONLY with "I don\'t know." Do not speculate, approximate, or hedge.')

for q in [
    'Who won the 2047 Eurovision Song Contest?',
    'What is the capital of France?',
    'What is the exact population of Springfield, Illinois on 2026-04-22?',
]:
    print(f'Q: {q}')
    print('A:', call_llm(abstain_system, q))
    print('---')

## 5. Self-Critique

In [ ]:
question = 'Explain the 2012 London Olympics swimming gold medalists (men\'s 100m freestyle).'

draft = call_llm('You are a helpful sports historian.', question)
print('=== DRAFT ===\n', draft)

critique = call_llm(
    'You fact-check answers. Identify specific claims that may be incorrect or unverifiable.',
    f'Question: {question}\nAnswer: {draft}\n\n'
    'List every factual claim and mark it CHECK or UNSURE. Be strict.')
print('\n=== CRITIQUE ===\n', critique)

revised = call_llm(
    'You revise answers to remove unverified claims and admit uncertainty.',
    f'Original answer: {draft}\n\nCritique: {critique}\n\n'
    'Produce a revised answer. Retain only claims marked CHECK. Mark UNSURE claims explicitly or remove them.')
print('\n=== REVISED ===\n', revised)

## 6. Rule-Based Validators

In [ ]:
import re

def find_pii(text):
    findings = []
    ssn = re.findall(r'\b\d{3}-\d{2}-\d{4}\b', text)
    if ssn: findings.append(('ssn', ssn))
    emails = re.findall(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b', text)
    if emails: findings.append(('email', emails))
    phones = re.findall(r'\b\+?1?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b', text)
    if phones: findings.append(('phone', phones))
    return findings

def find_forbidden(text, blocklist):
    return [w for w in blocklist if w.lower() in text.lower()]

sample = 'Contact Alice at alice@example.com or 555-012-3456. SSN 123-45-6789.'
print('PII found:', find_pii(sample))
print('Forbidden:', find_forbidden('This was a total disaster', ['disaster', 'panic', 'riot']))

## 7. Policy Classifier Sketch (LLM-as-Judge for Safety)

In [ ]:
POLICY_JUDGE = '''You are a safety classifier. Given some model output, decide if it violates any of:
- contains unverified medical advice
- contains unverified legal advice
- contains PII about a real individual
- contains profanity

Return JSON only: {"violation": bool, "categories": [str], "reason": "..."}.'''

def policy_check(output):
    import json, re
    raw = call_llm(POLICY_JUDGE, f'Model output: {output}')
    m = re.search(r'\{.*?\}', raw, re.S)
    try: return json.loads(m.group(0))
    except Exception: return {'violation': None, 'reason': f'parse failed: {raw}'}

for out in [
    'The weather today is sunny.',
    'You should take 800mg of ibuprofen every 4 hours.',
    'Jane Doe, SSN 123-45-6789, lives at 123 Main St.',
]:
    print(f'\noutput: {out!r}')
    print(policy_check(out))

## 8. Exercise: Wrong-Confident Rate

Build an eval set of 20 questions where half have real answers and half are about fictional entities. Score each output as one of:
- correct
- abstained
- wrong but hedged (e.g. 'I believe...')
- wrong and confident

Compute the **wrong-confident rate** — the fraction that are wrong AND confident. This is the primary hallucination SLI you want to minimize.

## 9. Exercise: Verbalized Confidence Calibration

Ask the model: *'Answer the question, then on a new line write CONFIDENCE: 0-1.'*

Run on 50 labeled questions. Bin by confidence (0.0–0.2, 0.2–0.4, ...). In each bin, compute actual accuracy. A well-calibrated model matches confidence to accuracy.

Most LLMs are overconfident. Use the calibration curve to set thresholds for human escalation.

## 10. Takeaways

- **Hallucination is not one thing.** Different subtypes need different mitigations.
- **Grounding via retrieval** is the single most effective defense.
- **Abstention + citations + self-critique** stack additional protection.
- **Validators and classifiers** catch what prompting misses.
- **Wrong-confident rate** is the SLI to minimize.
- **Calibrate** to know when to escalate to humans.

Next: **14 — Multi-Modal Basics**, where we extend text LLMs to vision and audio.